# Multi-Molecule GFlowNet For LPM24 On Colab (v2)

This notebook is the replay-sized Colab counterpart for LPM24 GFlowNet. It bootstraps the `gflownet_v2.4` branch in `/content/Thesis`, uses `scripts/init_colab.py` only for dependency installation, prepares the LPM24 grouped files with `scripts/download_lpm24.py` and `scripts/prepare_lpm24_training_splits.py`, then runs `scripts/train_multi_molecule_gflownet.py` directly with a temporary runtime config and exact Drive output directory. It supports `tb` and `db`, keeps the existing space-separated staged-target format, defaults to manual constrained beam-search rollout, uses fixed replay settings selected from the offline replay-sizing analysis, prefers the LPM24 multi-molecule SFT checkpoint, and falls back to the original BioT5 ChEBI-20 weights when that checkpoint is absent.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet_v2.4"
REPO_DIR = Path("/content/Thesis")

%cd /content
if (REPO_DIR / ".git").exists():
    print(f"Reusing {REPO_DIR}")
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", REPO_URL, REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", REPO_URL, REPO_BRANCH], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print({"repo_dir": str(REPO_DIR), "repo_branch": REPO_BRANCH})


In [ ]:
from datetime import datetime

DATASET_MODE = "never"
CONFIG_OVERRIDE = Path("configs/multi_molecule_gflownet_lpm24.yaml")
GFLOWNET_OBJECTIVE = "db"  # SubTB keeps only the latest valid stage per rollout.
DEFAULT_CONFIG_STEM = "multi_molecule_gflownet_lpm24"
COLAB_OUTPUT_ROOT = Path("/content/drive/MyDrive/multi_molecule_gflownet_lpm24")
RUN_STAMP = datetime.now().strftime("%y%m%d_%H%M%S")
RUN_NAME = f"{DEFAULT_CONFIG_STEM}_{GFLOWNET_OBJECTIVE}_{RUN_STAMP}"
COLAB_OUTPUT_DIR = COLAB_OUTPUT_ROOT / RUN_NAME
GFLOWNET_ITERATIONS = 35000
GFLOWNET_BATCH_SIZE = 4
MAX_OPTIMIZATION_TRAJECTORIES_PER_ITER = 42
GFLOWNET_SCORING_MICROBATCH_SIZE = 8
GFLOWNET_LEARNING_RATE = 1e-6
GFLOWNET_WARMUP_RATIO = 0.03
GFLOWNET_SAVE_EVERY_ITERATIONS = 500
GFLOWNET_INVALID_TERMINAL_REWARD = 4.0e-2
REWARD_PENALTY_INVALID = 0.5
VALIDATION_FRACTION = 0.05
SPLIT_SEED = 42
MAX_TARGET_SYMBOLS = 1024
# `MAX_STAGE_SYMBOLS` only filters samples in `scripts/prepare_lpm24_training_splits.py`.
# It does not change the rollout stop condition during GFlowNet sampling.
MAX_STAGE_SYMBOLS = 128
GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE = "1jCIVYbzgTw7xQAWvv6SfwM8Y1vL47PDg"
FORCE_VALID_MASKING = True
SELFIES_DICT_PATH = "molecules/dict/selfies_dict.txt"
# Rollout controls: per-stage token cap, planned stage count, then overall sequence cap.
ROLLOUT_MAX_STAGE_NEW_TOKENS = 128
ROLLOUT_MAX_MOLECULES_PER_SEQUENCE = 6
ROLLOUT_MAX_SEQUENCE_LENGTH = 8192
ROLLOUT_DECODING_STRATEGY = "beam"
ROLLOUT_NUM_BEAMS = 2
ROLLOUT_LENGTH_PENALTY = 1.0
ROLLOUT_EARLY_STOPPING = True
# Target-guided off-policy controls: fixed 25/50/25 source mix.
TARGET_GUIDANCE_ENABLED = True
TARGET_GUIDANCE_ON_POLICY_FRACTION = 0.25
TARGET_GUIDANCE_PREFIX_FRACTION = 0.50
TARGET_GUIDANCE_TEACHER_FRACTION = 0.25
TARGET_GUIDANCE_SHUFFLE_TARGET_SELFIES_LIST = True
# Replay is disabled by default when target guidance is enabled.
REPLAY_ENABLED = False
REPLAY_BUFFER_TYPE = "experimental_mixture"
REPLAY_FRACTION = 0.75
REPLAY_WITH_REPLACEMENT = True
REPLAY_CAPACITY = 157772
# Kept for config compatibility; beam rollout is controlled by the beam settings above.
ROLLOUT_TEMPERATURE = 0.16
ROLLOUT_TOP_P = 0.55
ROLLOUT_APPEND_PROBABILITY = 1.0
ROLLOUT_INVALID_APPEND_PROBABILITY = 0.75
TERMINATE_ON_INVALID_STAGE = True
# Reward shaping: medium-low diversity pressure for the first rerun.
REWARD_DIVERSITY_BETA = 0.6
REWARD_DIVERSITY_WEIGHT = 0.4
REWARD_MATCH_WEIGHT = 1.0
REWARD_VARIANT = "reward_var2"
# reward_v1: REWARD_VARIANT = "reward_var1"
REWARD_PLUS_VALID = 0.4
ENABLE_INVALID_SIMILARITY_NGRAM_FALLBACK = False

BASE_TRAIN_CONFIG = CONFIG_OVERRIDE or Path("configs") / f"{DEFAULT_CONFIG_STEM}.yaml"
WANDB_PROJECT_URL = "https://wandb.ai/koala-team/Thesis-2"

OUTPUT_DIR = COLAB_OUTPUT_DIR.expanduser()
OUTPUT_DIR_ZIP = OUTPUT_DIR / f"{OUTPUT_DIR.name}.zip"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
BEST_CHECKPOINT_DIR = CHECKPOINTS_DIR / "best"
BEST_CHECKPOINT_ZIP = CHECKPOINTS_DIR / "best.zip"
RUN_SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"
ITERATION_DIAGNOSTICS_PATH = OUTPUT_DIR / "diagnostics" / "iteration_diagnostics.jsonl"
TRAJECTORY_PREVIEWS_PATH = OUTPUT_DIR / "diagnostics" / "trajectory_previews.jsonl"
GFLOWNET_REPORT_METRICS_PATH = OUTPUT_DIR / "diagnostics" / "gflownet_report_metrics.jsonl"
TEMP_CONFIG_PATH = REPO_DIR / "tmp" / "colab_runtime" / f"{BASE_TRAIN_CONFIG.stem}_{GFLOWNET_OBJECTIVE}_06_v2.yaml"
DEFAULT_PPO_FALLBACK_CHECKPOINT = "QizhiPei/biot5-plus-base-chebi20"
UPSTREAM_CHECKPOINT = REPO_DIR / "outputs" / "multi_molecule_sft_lpm24" / "checkpoints" / "best"
LPM24_DATASET_DIR = REPO_DIR / "data" / "lpm24"
GROUPED_SPLITS_DIR = LPM24_DATASET_DIR / "grouped_splits"
PROCESSED_DATASET_CHECKS = {
    "train_multimol": LPM24_DATASET_DIR / "processed" / "train_multimol.jsonl",
    "test_multimol": LPM24_DATASET_DIR / "processed" / "test_multimol.jsonl",
    "test_eval_first_1000": LPM24_DATASET_DIR / "processed" / "test_eval_first_1000_multimol.jsonl",
}
GROUPED_SPLIT_CHECKS = {
    "train_multimol": GROUPED_SPLITS_DIR / "train_multimol.jsonl",
    "validation_multimol": GROUPED_SPLITS_DIR / "validation_multimol.jsonl",
    "test_multimol": GROUPED_SPLITS_DIR / "test_multimol.jsonl",
}

print({
    "base_train_config": str(BASE_TRAIN_CONFIG),
    "colab_output_root": str(COLAB_OUTPUT_ROOT),
    "run_name": RUN_NAME,
    "colab_output_dir": str(COLAB_OUTPUT_DIR),
    "gflownet_objective": GFLOWNET_OBJECTIVE,
    "gflownet_iterations": GFLOWNET_ITERATIONS,
    "gflownet_batch_size": GFLOWNET_BATCH_SIZE,
    "max_optimization_trajectories_per_iter": MAX_OPTIMIZATION_TRAJECTORIES_PER_ITER,
    "gflownet_scoring_microbatch_size": GFLOWNET_SCORING_MICROBATCH_SIZE,
    "gflownet_learning_rate": GFLOWNET_LEARNING_RATE,
    "gflownet_warmup_ratio": GFLOWNET_WARMUP_RATIO,
    "gflownet_invalid_terminal_reward": GFLOWNET_INVALID_TERMINAL_REWARD,
    "reward_penalty_invalid": REWARD_PENALTY_INVALID,
    "validation_fraction": VALIDATION_FRACTION,
    "split_seed": SPLIT_SEED,
    "max_target_symbols": MAX_TARGET_SYMBOLS,
    "max_stage_symbols": MAX_STAGE_SYMBOLS,
    "gflownet_checkpoint_download_source_configured": bool(GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip()),
    "force_valid_masking": FORCE_VALID_MASKING,
    "selfies_dict_path": SELFIES_DICT_PATH,
    "rollout_max_stage_new_tokens": ROLLOUT_MAX_STAGE_NEW_TOKENS,
    "rollout_max_molecules_per_sequence": ROLLOUT_MAX_MOLECULES_PER_SEQUENCE,
    "rollout_max_sequence_length": ROLLOUT_MAX_SEQUENCE_LENGTH,
    "rollout_decoding_strategy": ROLLOUT_DECODING_STRATEGY,
    "rollout_num_beams": ROLLOUT_NUM_BEAMS,
    "rollout_length_penalty": ROLLOUT_LENGTH_PENALTY,
    "rollout_early_stopping": ROLLOUT_EARLY_STOPPING,
    "target_guidance_enabled": TARGET_GUIDANCE_ENABLED,
    "target_guidance_on_policy_fraction": TARGET_GUIDANCE_ON_POLICY_FRACTION,
    "target_guidance_prefix_fraction": TARGET_GUIDANCE_PREFIX_FRACTION,
    "target_guidance_teacher_fraction": TARGET_GUIDANCE_TEACHER_FRACTION,
    "target_guidance_shuffle_target_selfies_list": TARGET_GUIDANCE_SHUFFLE_TARGET_SELFIES_LIST,
    "replay_enabled": REPLAY_ENABLED,
    "replay_buffer_type": REPLAY_BUFFER_TYPE,
    "replay_fraction": REPLAY_FRACTION,
    "replay_with_replacement": REPLAY_WITH_REPLACEMENT,
    "replay_capacity": REPLAY_CAPACITY,
    "rollout_top_p": ROLLOUT_TOP_P,
    "rollout_temperature": ROLLOUT_TEMPERATURE,
    "rollout_append_probability": ROLLOUT_APPEND_PROBABILITY,
    "rollout_invalid_append_probability": ROLLOUT_INVALID_APPEND_PROBABILITY,
    "terminate_on_invalid_stage": TERMINATE_ON_INVALID_STAGE,
    "reward_diversity_beta": REWARD_DIVERSITY_BETA,
    "reward_diversity_weight": REWARD_DIVERSITY_WEIGHT,
    "reward_match_weight": REWARD_MATCH_WEIGHT,
    "reward_variant": REWARD_VARIANT,
    "reward_plus_valid": REWARD_PLUS_VALID,
    "enable_invalid_similarity_ngram_fallback": ENABLE_INVALID_SIMILARITY_NGRAM_FALLBACK,
    "output_dir": str(OUTPUT_DIR),
})


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print({
    "colab_output_root": str(COLAB_OUTPUT_ROOT),
    "run_name": RUN_NAME,
    "colab_output_dir": str(COLAB_OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
})


In [ ]:
import os
import wandb

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY, relogin=True)

print({
    "wandb_api_key_configured": bool(WANDB_API_KEY),
    "wandb_project_url": WANDB_PROJECT_URL,
})


In [ ]:
%cd {REPO_DIR}
bootstrap_command = [
    sys.executable,
    "scripts/init_colab.py",
    "--stage",
    "gflownet",
    "--repo-url",
    REPO_URL,
    "--repo-branch",
    REPO_BRANCH,
    "--repo-dir",
    str(REPO_DIR),
    "--dataset-mode",
    DATASET_MODE,
]
if CONFIG_OVERRIDE:
    bootstrap_command.extend(["--config", str(CONFIG_OVERRIDE)])
if GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip():
    bootstrap_command.extend([
        "--gflownet-checkpoint-download-source",
        GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip(),
    ])

print("Bootstrapping:", " ".join(str(part) for part in bootstrap_command))

def run_and_stream(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Failed to capture command output.")

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

run_and_stream(bootstrap_command)

processed_dataset_ready = all(path.exists() for path in PROCESSED_DATASET_CHECKS.values())
if not processed_dataset_ready:
    download_command = [
        sys.executable,
        "scripts/download_lpm24.py",
        "--output-dir",
        str(LPM24_DATASET_DIR),
    ]
    print("Preparing grouped LPM24 dataset:", " ".join(str(part) for part in download_command))
    run_and_stream(download_command)
else:
    print("Reusing grouped LPM24 dataset:", str(LPM24_DATASET_DIR))

export_command = [
    sys.executable,
    "scripts/prepare_lpm24_training_splits.py",
    "--input-dir",
    str(LPM24_DATASET_DIR),
    "--validation-fraction",
    str(VALIDATION_FRACTION),
    "--seed",
    str(SPLIT_SEED),
    "--max-target-symbols",
    str(MAX_TARGET_SYMBOLS),
    "--max-stage-symbols",
    str(MAX_STAGE_SYMBOLS),
]
print("Preparing training-ready LPM24 splits:", " ".join(str(part) for part in export_command))
run_and_stream(export_command)

import yaml

TEMP_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
runtime_config = yaml.safe_load(BASE_TRAIN_CONFIG.read_text())
runtime_config.setdefault("gflownet", {})["objective"] = GFLOWNET_OBJECTIVE

gflownet_payload = runtime_config.setdefault("gflownet", {})
gflownet_payload["gflownet_iterations"] = int(GFLOWNET_ITERATIONS)
gflownet_payload["batch_size"] = int(GFLOWNET_BATCH_SIZE)
gflownet_payload["max_optimization_trajectories_per_iter"] = int(MAX_OPTIMIZATION_TRAJECTORIES_PER_ITER)
gflownet_payload["scoring_microbatch_size"] = int(GFLOWNET_SCORING_MICROBATCH_SIZE)
gflownet_payload["learning_rate"] = float(GFLOWNET_LEARNING_RATE)
gflownet_payload["warmup_ratio"] = float(GFLOWNET_WARMUP_RATIO)
gflownet_payload["save_every_iterations"] = int(GFLOWNET_SAVE_EVERY_ITERATIONS)
gflownet_payload["invalid_terminal_reward"] = float(GFLOWNET_INVALID_TERMINAL_REWARD)
target_guidance_payload = dict(gflownet_payload.get("target_guidance", {}))
target_guidance_payload["enabled"] = bool(TARGET_GUIDANCE_ENABLED)
target_guidance_payload["on_policy_fraction"] = float(TARGET_GUIDANCE_ON_POLICY_FRACTION)
target_guidance_payload["target_prefix_rollout_fraction"] = float(TARGET_GUIDANCE_PREFIX_FRACTION)
target_guidance_payload["target_teacher_fraction"] = float(TARGET_GUIDANCE_TEACHER_FRACTION)
target_guidance_payload["shuffle_target_selfies_list"] = bool(TARGET_GUIDANCE_SHUFFLE_TARGET_SELFIES_LIST)
gflownet_payload["target_guidance"] = target_guidance_payload
replay_payload = dict(gflownet_payload.get("replay", {}))
replay_payload["enabled"] = bool(REPLAY_ENABLED)
replay_payload["buffer_type"] = REPLAY_BUFFER_TYPE
replay_payload["replay_fraction"] = float(REPLAY_FRACTION)
replay_payload["with_replacement"] = bool(REPLAY_WITH_REPLACEMENT)
replay_payload["capacity"] = int(REPLAY_CAPACITY)
gflownet_payload["replay"] = replay_payload
rollout_payload = dict(gflownet_payload.get("rollout", {}))
rollout_payload["max_stage_new_tokens"] = int(ROLLOUT_MAX_STAGE_NEW_TOKENS)
rollout_payload["max_molecules_per_sequence"] = int(ROLLOUT_MAX_MOLECULES_PER_SEQUENCE)
rollout_payload["max_sequence_length"] = int(ROLLOUT_MAX_SEQUENCE_LENGTH)
rollout_payload["decoding_strategy"] = ROLLOUT_DECODING_STRATEGY
rollout_payload["num_beams"] = int(ROLLOUT_NUM_BEAMS)
rollout_payload["length_penalty"] = float(ROLLOUT_LENGTH_PENALTY)
rollout_payload["early_stopping"] = bool(ROLLOUT_EARLY_STOPPING)
if FORCE_VALID_MASKING:
    rollout_payload["constrained_decoding"] = True
    rollout_payload["selfies_dict_path"] = SELFIES_DICT_PATH
rollout_payload["stage_separator"] = " "
if ROLLOUT_TEMPERATURE is not None:
    rollout_payload["temperature"] = float(ROLLOUT_TEMPERATURE)
if ROLLOUT_TOP_P is not None:
    rollout_payload["top_p"] = float(ROLLOUT_TOP_P)
rollout_payload["append_probability"] = float(ROLLOUT_APPEND_PROBABILITY)
rollout_payload["invalid_append_probability"] = float(ROLLOUT_INVALID_APPEND_PROBABILITY)
rollout_payload["terminate_on_invalid_stage"] = bool(TERMINATE_ON_INVALID_STAGE)
gflownet_payload["rollout"] = rollout_payload
reward_payload = dict(runtime_config.get("reward", {}))
reward_payload["diversity_beta"] = float(REWARD_DIVERSITY_BETA)
reward_payload["diversity_weight"] = float(REWARD_DIVERSITY_WEIGHT)
reward_payload["match_weight"] = float(REWARD_MATCH_WEIGHT)
reward_payload["reward_variant"] = REWARD_VARIANT
reward_payload["plus_valid"] = float(REWARD_PLUS_VALID)
reward_payload["penalty_invalid"] = float(REWARD_PENALTY_INVALID)
reward_payload["enable_invalid_similarity_ngram_fallback"] = bool(ENABLE_INVALID_SIMILARITY_NGRAM_FALLBACK)
runtime_config["reward"] = reward_payload

TEMP_CONFIG_PATH.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding="utf-8")
control_summary = {
    "runtime_config": str(TEMP_CONFIG_PATH),
    "colab_output_dir": str(COLAB_OUTPUT_DIR),
    "gflownet_objective": GFLOWNET_OBJECTIVE,
    "gflownet_iterations": gflownet_payload.get("gflownet_iterations"),
    "gflownet_batch_size": gflownet_payload.get("batch_size"),
    "max_optimization_trajectories_per_iter": gflownet_payload.get("max_optimization_trajectories_per_iter"),
    "gflownet_scoring_microbatch_size": gflownet_payload.get("scoring_microbatch_size"),
    "gflownet_learning_rate": gflownet_payload.get("learning_rate"),
    "gflownet_warmup_ratio": gflownet_payload.get("warmup_ratio"),
    "gflownet_save_every_iterations": gflownet_payload.get("save_every_iterations"),
    "gflownet_invalid_terminal_reward": gflownet_payload.get("invalid_terminal_reward"),
    "validation_fraction": VALIDATION_FRACTION,
    "split_seed": SPLIT_SEED,
    "max_target_symbols": MAX_TARGET_SYMBOLS,
    "max_stage_symbols": MAX_STAGE_SYMBOLS,
    "force_valid_masking": FORCE_VALID_MASKING,
    "rollout_max_stage_new_tokens": rollout_payload.get("max_stage_new_tokens"),
    "rollout_max_molecules_per_sequence": rollout_payload.get("max_molecules_per_sequence"),
    "rollout_max_sequence_length": rollout_payload.get("max_sequence_length"),
    "rollout_decoding_strategy": rollout_payload.get("decoding_strategy"),
    "rollout_num_beams": rollout_payload.get("num_beams"),
    "rollout_length_penalty": rollout_payload.get("length_penalty"),
    "rollout_early_stopping": rollout_payload.get("early_stopping"),
    "rollout_temperature": rollout_payload.get("temperature"),
    "rollout_top_p": rollout_payload.get("top_p"),
    "rollout_append_probability": rollout_payload.get("append_probability"),
    "rollout_invalid_append_probability": rollout_payload.get("invalid_append_probability"),
    "terminate_on_invalid_stage": rollout_payload.get("terminate_on_invalid_stage"),
    "target_guidance_enabled": target_guidance_payload.get("enabled"),
    "target_guidance_on_policy_fraction": target_guidance_payload.get("on_policy_fraction"),
    "target_guidance_prefix_fraction": target_guidance_payload.get("target_prefix_rollout_fraction"),
    "target_guidance_teacher_fraction": target_guidance_payload.get("target_teacher_fraction"),
    "target_guidance_shuffle_target_selfies_list": target_guidance_payload.get("shuffle_target_selfies_list"),
    "replay_enabled": replay_payload.get("enabled"),
    "replay_buffer_type": replay_payload.get("buffer_type"),
    "replay_fraction": replay_payload.get("replay_fraction"),
    "replay_with_replacement": replay_payload.get("with_replacement"),
    "replay_capacity": replay_payload.get("capacity"),
    "reward_diversity_beta": reward_payload.get("diversity_beta"),
    "reward_diversity_weight": reward_payload.get("diversity_weight"),
    "reward_match_weight": reward_payload.get("match_weight"),
    "reward_variant": reward_payload.get("reward_variant"),
    "reward_plus_valid": reward_payload.get("plus_valid"),
    "reward_penalty_invalid": reward_payload.get("penalty_invalid"),
    "enable_invalid_similarity_ngram_fallback": reward_payload.get("enable_invalid_similarity_ngram_fallback"),
    "constrained_decoding": rollout_payload.get("constrained_decoding"),
    "selfies_dict_path": rollout_payload.get("selfies_dict_path"),
    "stage_separator": rollout_payload.get("stage_separator"),
}
print(control_summary)
print(
    "Target guidance: this notebook trains with fixed 25/50/25 on-policy, "
    "target-prefix rollout, and target-teacher source fractions; replay is disabled "
    "unless explicitly re-enabled with target guidance off."
)
print(
    "Tuning guide: `MAX_STAGE_SYMBOLS` filters dataset export, `max_stage_new_tokens` caps each "
    "rollout action, `max_molecules_per_sequence` caps planned stage actions, `max_sequence_length` "
    "caps the overall encoder+decoder length, `max_optimization_trajectories_per_iter` caps "
    "scored training trajectories, `scoring_microbatch_size` caps padded gradient scoring "
    "microbatches, and target guidance adds valid-prefix "
    "off-policy stage trajectories to the optimization batch."
)
print(
    "`termination_fraction_max_stage_new_tokens` is the external/ `max-stage` metric: it rises when "
    "rollout hits `gflownet.rollout.max_stage_new_tokens` before emitting `<EOM>` or `<eos>`."
)
print(
    "After each run, inspect `mean_terminal_stop_logprob`, `valid_fraction`, and "
    "`termination_fraction_stop_token` together to judge stop behavior."
)
if GFLOWNET_OBJECTIVE == "subtb":
    print("SubTB retention: only the latest valid stage trajectory is kept per rollout.")


if not UPSTREAM_CHECKPOINT.exists():
    print(
        "Default upstream checkpoint is missing at "
        f"{UPSTREAM_CHECKPOINT}; GFlowNet will fall back to {DEFAULT_PPO_FALLBACK_CHECKPOINT}."
    )

training_command = [
    sys.executable,
    "scripts/train_multi_molecule_gflownet.py",
    "--config",
    str(TEMP_CONFIG_PATH),
    "--output-dir",
    str(OUTPUT_DIR),
]
print("Training:", " ".join(str(part) for part in training_command))
run_and_stream(training_command)


In [ ]:
import json
import random

import torch
import wandb
import yaml
from transformers import AutoTokenizer

from evaluation_metrics import (
    EvaluationMetricConfig,
    GenerationGroup,
    MoleculeInput,
    evaluate_generation_groups,
)
from post_training.gflownet import (
    GFlowNetModel,
    build_gflownet_config,
    build_reward_config,
    sample_stage_trajectories_for_example,
)
from post_training.shared.dataset import MultiMoleculeDataset
from post_training.shared.decoding import build_stage_token_constraints
from src.device import choose_device

EVAL_NUM_EXAMPLES = 128
EVAL_SEED = 42
EVAL_OUTPUT_PATH = OUTPUT_DIR / "diagnostics" / "validation_evaluation_metrics.json"

validation_dataset = MultiMoleculeDataset.from_jsonl(GROUPED_SPLITS_DIR / "validation_multimol.jsonl")
rng = random.Random(EVAL_SEED)
eval_examples = [
    validation_dataset[rng.randrange(len(validation_dataset))]
    for _ in range(min(EVAL_NUM_EXAMPLES, len(validation_dataset)))
]

summary_payload = json.loads(RUN_SUMMARY_PATH.read_text()) if RUN_SUMMARY_PATH.exists() else {}
best_checkpoint_dir = Path(summary_payload["best_checkpoint_dir"]) if summary_payload.get("best_checkpoint_dir") else BEST_CHECKPOINT_DIR
checkpoint_for_eval = best_checkpoint_dir if best_checkpoint_dir.exists() else BEST_CHECKPOINT_DIR

tokenizer = AutoTokenizer.from_pretrained(checkpoint_for_eval, use_fast=True)
tokenizer.model_max_length = int(1e9)

eval_config = yaml.safe_load(TEMP_CONFIG_PATH.read_text())
gflownet_config = build_gflownet_config(eval_config)
reward_config = build_reward_config(
    eval_config.get("reward", {}),
    dataset_hint=str(eval_config.get("data", {}).get("validation_file", "")),
)

device = choose_device(eval_config["training"].get("device", "auto"))
model = GFlowNetModel.from_pretrained(
    checkpoint_for_eval,
    use_lora=gflownet_config.use_lora,
    lora_rank=gflownet_config.lora_rank,
    lora_alpha=gflownet_config.lora_alpha,
    lora_dropout=gflownet_config.lora_dropout,
    target_modules=gflownet_config.target_modules,
    freeze_base_model_without_lora=gflownet_config.freeze_base_model_without_lora,
)
model.to(device)
model.eval()

if gflownet_config.rollout.constrained_decoding:
    constraints = build_stage_token_constraints(
        tokenizer,
        validation_dataset,
        selfies_dict_path=gflownet_config.rollout.selfies_dict_path,
        separator_token=gflownet_config.rollout.stage_separator,
    )
    model.set_stage_token_constraints(constraints)

groups = []
with torch.no_grad():
    for index, example in enumerate(eval_examples):
        trajectories = sample_stage_trajectories_for_example(
            model,
            tokenizer,
            example,
            rollout_id=f"validation-eval-{index:06d}-{example['id']}",
            generation_config=gflownet_config.rollout,
            reward_config=reward_config,
            invalid_terminal_reward=gflownet_config.invalid_terminal_reward,
            device=device,
            rng=rng,
            return_last_valid_trajectory_only=False,
        )

        generated_selfies = [
            trajectory.sampled_selfies
            for trajectory in trajectories
            if trajectory.sampled_selfies
        ]

        groups.append(
            GenerationGroup(
                group_id=str(example["id"]),
                candidates=tuple(
                    MoleculeInput(text=selfies, representation="selfies")
                    for selfies in generated_selfies
                ),
                targets=tuple(
                    MoleculeInput(text=selfies, representation="selfies")
                    for selfies in example["target_selfies_list"]
                ),
            )
        )

result = evaluate_generation_groups(
    groups,
    config=EvaluationMetricConfig(
        acceptance_dice_threshold=0.7,
        compute_n_circles=False,
        n_circles_tanimoto_threshold=0.6,
    ),
)

payload = result.to_dict(include_assessments=False)
EVAL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")

evaluation_diagnosis = {
    "eval/accepted_unique_count": result.accepted_unique_count,
    "eval/n_circles": result.n_circles,
    "eval/internal_diversity": result.internal_diversity,
    "eval/novelty_fraction": result.novelty_fraction,
    "eval/novelty_count": result.novelty_count,
    "eval/valid_fraction": result.num_valid_candidates / max(result.num_candidates, 1),
    "eval/mean_max_dice_similarity": result.mean_max_dice_similarity,
}

tracking_config = eval_config.get("tracking", {})
if wandb.run is None and tracking_config.get("enabled", True):
    wandb.init(
        project=tracking_config.get("project", "Thesis-2"),
        entity=tracking_config.get("workspace") or None,
        name=f"{RUN_NAME}-validation-eval",
        job_type="validation_evaluation",
    )
wandb.log(evaluation_diagnosis)
print(evaluation_diagnosis)
print({"validation_evaluation_metrics_path": str(EVAL_OUTPUT_PATH)})


In [ ]:
import json

from src.checkpoint_bootstrap import archive_directory_to_zip

summary_payload = json.loads(RUN_SUMMARY_PATH.read_text()) if RUN_SUMMARY_PATH.exists() else {}
best_checkpoint_dir = Path(summary_payload["best_checkpoint_dir"]) if summary_payload.get("best_checkpoint_dir") else BEST_CHECKPOINT_DIR
best_checkpoint_zip = Path(summary_payload["best_checkpoint_zip"]) if summary_payload.get("best_checkpoint_zip") else BEST_CHECKPOINT_ZIP
output_dir_zip = None
if OUTPUT_DIR.exists():
    output_dir_zip = archive_directory_to_zip(OUTPUT_DIR, OUTPUT_DIR_ZIP)
if best_checkpoint_dir.exists() and not best_checkpoint_zip.exists():
    best_checkpoint_zip = archive_directory_to_zip(best_checkpoint_dir, best_checkpoint_zip)

print({
    "gflownet_objective": GFLOWNET_OBJECTIVE,
    "output_dir": str(OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "output_dir_zip": str(output_dir_zip) if output_dir_zip else str(OUTPUT_DIR_ZIP),
    "output_dir_zip_exists": output_dir_zip is not None and output_dir_zip.exists(),
    "checkpoint_dir": str(CHECKPOINTS_DIR),
    "checkpoint_dir_exists": CHECKPOINTS_DIR.exists(),
    "saved_iteration_checkpoints": sorted(path.name for path in CHECKPOINTS_DIR.glob("iteration-*")) if CHECKPOINTS_DIR.exists() else [],
    "best_checkpoint_dir": str(best_checkpoint_dir),
    "best_checkpoint_dir_exists": best_checkpoint_dir.exists(),
    "best_checkpoint_zip": str(best_checkpoint_zip),
    "best_checkpoint_zip_exists": best_checkpoint_zip.exists(),
    "run_summary": str(RUN_SUMMARY_PATH),
    "run_summary_exists": RUN_SUMMARY_PATH.exists(),
    "best_objective_loss": summary_payload.get("best_objective_loss"),
    "best_checkpoint_iteration": summary_payload.get("best_checkpoint_iteration"),
    "resolved_checkpoint_source": summary_payload.get("resolved_checkpoint_source"),
    "iteration_diagnostics_path": str(ITERATION_DIAGNOSTICS_PATH),
    "iteration_diagnostics_exists": ITERATION_DIAGNOSTICS_PATH.exists(),
    "trajectory_previews_path": str(TRAJECTORY_PREVIEWS_PATH),
    "trajectory_previews_exists": TRAJECTORY_PREVIEWS_PATH.exists(),
    "gflownet_report_metrics_path": str(GFLOWNET_REPORT_METRICS_PATH),
    "gflownet_report_metrics_exists": GFLOWNET_REPORT_METRICS_PATH.exists(),
    "upstream_checkpoint": str(UPSTREAM_CHECKPOINT),
    "upstream_checkpoint_exists": UPSTREAM_CHECKPOINT.exists(),
    "lpm24_dataset_dir": str(LPM24_DATASET_DIR),
    "lpm24_dataset_exists": LPM24_DATASET_DIR.exists(),
    "processed_dataset_checks": {name: path.exists() for name, path in PROCESSED_DATASET_CHECKS.items()},
    "grouped_split_checks": {name: path.exists() for name, path in GROUPED_SPLIT_CHECKS.items()},
})
